#BehaviorScoringRun.ipynb
🪰 This notebook is the entry point for the **BehaviorScoring** pipeline.
<br><br>
It keeps implementation details in modules and streamlines the workflow: set up the environment, select paths, run the analysis, and produce consistent scored outputs.  
The emphasis is a user-friendly workflow built on standardized, reproducible practices, i.e. clear separation of config from logic, minimal editing in one place, and version-pinned runs.
<br><br>

---
#### Overview
```
1. Prepare the environment.  
2. (Temporary) point to the code folder and experiment root.  
3. Load the pipeline modules and configuration.  
4. Run the main behavior scoring pipeline across the experiment data.  
5. Save scored files and logs to the predefined output folders.
```
---
#### Dependencies
```
BehaviorScoringMain.py
BehaviorScoringFunctions.py
BehaviorScoringConfig.py
BehaviorScoringColabConfig.py
ExperimentConfig.py
PathConfig.py
```

##Setup enviroment
1. Pin dependency versions
2. Mount Google Drive
3. Set `pPathConfig` to the `PathConfig.py` path

In [ ]:
# Pin & install dependencies
#python==3.11.13
!pip install pandas==2.3.1 numpy==2.3.2 scikit-learn==1.7.1 scipy==1.16.1

from pathlib import Path
import shutil
import sys, types, importlib

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Placeholder for your PathConfig location
#    Once you remove the TEMP cell, simply uncomment and edit:
# pPathConfig = Path("/content/drive/My Drive/YourExperimentFolder/code/PathConfig.py")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#%% TEMP — minimal bootstrap (safe to delete once PathConfig has the real root)

# YOU set these two:
pPathConfig = Path("/content/drive/My Drive/FHAnalysisPipeline/306DB/Protocols/Codes/Config/PathConfig.py")
tempExperimentalRoot = Path("/content/drive/My Drive/FHAnalysisPipeline/306DB")

# 1) Inject the root into PathConfig.py IN MEMORY (no disk changes)
code = pPathConfig.read_text()
code = code.replace('"__EXP_ROOT__"', f'"{tempExperimentalRoot.as_posix()}"')

PathConfig = types.ModuleType("PathConfig")
exec(compile(code, str(pPathConfig), "exec"), PathConfig.__dict__)

# 2) Expose package root (Codes) for dotted imports
sys.path.insert(0, str(PathConfig.pCodes))

# 3) Ensure any future imports get THIS injected module
sys.modules["PathConfig"] = PathConfig
importlib.import_module("Config")                 # load package once
sys.modules["Config.PathConfig"] = PathConfig     # satisfy 'from Config import PathConfig'


##Config and Stage
1. Load pipeline configuration
2. Prepare a fast local workspace.

In [ ]:
from Config import ExperimentConfig
from Config import BehaviorScoringConfig
from Config import BehaviorScoringColabConfig

# 1) Read canonical Drive paths from PathConfig
drive_root, drive_paths = BehaviorScoringColabConfig.load_configs(PathConfig)

# 2) Always clean local workspace for this experiment (fresh start)
local_base = Path("/content/exp_runs") / drive_root.name
if local_base.exists(): shutil.rmtree(local_base, ignore_errors=True)

# 4) Validate required inputs (strict pose rule)
BehaviorScoringColabConfig.validate_inputs(drive_paths)

# 5) Build local mirrors under /content (will recreate the cleaned folders)
local_paths = BehaviorScoringColabConfig.local_mirrors(drive_root, drive_paths)

# 6) Stage inputs + existing outputs to local (preserves skip logic)
staged = BehaviorScoringColabConfig.stage_to_local(drive_paths, local_paths)


================= LOADING FILES FROM DRIVE =================

   Estimated: ~00:53 at 104.8 MB/s for 488/612 files

   Inputs        : Tracked=244, Pose=244
   Existing outs : Scored=59, ScoredError=3
   Staging time  : 01:26

======================= READY TO RUN =======================


##Import and run the pipeline
1. Make codes folder importable.
2. Import modules.
3. Call `BehaviorScoringMain` to process all files.

In [ ]:
# Edit this to your liking:
BATCH_SYNC_SIZE = 50     # Save scored to drive in batch - tracked + pose counts as 2 files

# Import modules from Drive
from BehaviorScoring import BehaviorScoringFunctions
from BehaviorScoring.BehaviorScoringMain import behavior_scoring_main

# Build temporary PathConfig pointing to local mirrors
LocalPathConfig = BehaviorScoringColabConfig.make_local_pathconfig(PathConfig, local_paths)

# Start silent background sync
BehaviorScoringColabConfig.start_background_sync(local_paths, drive_paths, batch_size=BATCH_SYNC_SIZE)

# Run the pipeline
behavior_scoring_main(LocalPathConfig, ExperimentConfig, BehaviorScoringConfig, BehaviorScoringFunctions)

# Stop background sync and do a final silent push
BehaviorScoringColabConfig.stop_background_sync()
BehaviorScoringColabConfig.sync_outputs_back(local_paths, drive_paths)

# Optional: clean up local workspace
# import shutil
# shutil.rmtree(local_paths.root, ignore_errors=True)
# ============================================================================


PROCESSING: /content/exp_runs/306DB
POSE:    TRUE

FILES FOUND   -----------------------------------------   306
TO SCORE   --------------------------------------------   244
SKIPPING   ---------------------------------------------   62
---   scored: 59   ---   errors: 3   ---


SCORING: file 1/244 (0.00 s/file – 00h00 eta)  |  CR-EmptyTNT-20Control_3BlackOut-Female-6days-FH3-CamA-2024-07-22T14_59_13_fly0
SCORING: file 2/244 (22.66 s/file – 01h54 eta)  |  CR-EmptyTNT-20Control_3BlackOut-Female-6days-FH3-CamA-2024-07-22T14_59_13_fly1
SCORING: file 3/244 (20.54 s/file – 01h43 eta)  |  CR-EmptyTNT-20Control_3BlackOut-Female-6days-FH3-CamA-2024-07-22T14_59_13_fly2
SCORING: file 4/244 (20.72 s/file – 01h44 eta)  |  CR-EmptyTNT-20Control_3BlackOut-Female-6days-FH3-CamA-2024-07-22T14_59_13_fly3
SCORING: file 5/244 (21.31 s/file – 01h46 eta)  |  CR-EmptyTNT-20Control_3BlackOut-Female-6days-FH3-CamA-2024-07-22T18_00_48_fly0
SCORING: file 6/244 (21.39 s/file – 01h46 eta)  |  CR-EmptyTNT-20Contro

KeyboardInterrupt: 